# Lotka-Volterra Equations (Two-Species System)

The standard two-species LV model is given by the system of ODEs:

$$
\frac{dx}{dt} = \alpha x - \beta x y
$$
$$
\frac{dy}{dt} = \delta x y - \gamma y
$$

where  
- $ x(t) $ is the prey population at time $ t $.  
- $ y(t) $ is the predator population at time $ t $.  

The parameters $\alpha, \beta, \gamma, \delta$ are all positive constants:
- $\alpha$: intrinsic growth rate of the prey (in absence of predators).  
- $\beta$: predation rate coefficient (how effectively predators consume prey).  
- $\gamma$: predator mortality rate (in absence of prey).  
- $\delta$: efficiency by which consumed prey is converted into predator growth.

**Causal Interpretation:**
- The prey population $ x $ grows due to $\alpha x$ but is reduced by interactions with predators $ -\beta x y $. Thus, predator presence causes a decline in prey.  
- The predator population $ y $ decays at rate $\gamma$ but increases through predation on the prey $ \delta x y $. Thus, the prey population causes an increase in the predator population.

Causally:
- $ x $ (prey) influences $ y $ (predator).
- $ y $ (predator) influences $ x $ (prey).

This is a bidirectionally coupled system, but the direction of influence is encoded in the functional form: prey growth is not directly driven by predator population except negatively (predators only reduce it), while predator growth depends positively on prey. If you introduced additional species or states, you could create more complex causal structures.


## Bivariate Case

In [1]:
import numpy as np
from scipy.integrate import solve_ivp

# Define parameters
alpha, beta, gamma, delta = 1.0, 0.1, 1.0, 0.075

def lotka_volterra(t, z):
    x, y = z
    dxdt = alpha*x - beta*x*y
    dydt = delta*x*y - gamma*y
    return [dxdt, dydt]

# Time span and initial conditions
t_span = (0, 50)
z0 = [10, 5]  # initial conditions for x and y

# Solve ODE
sol = solve_ivp(lotka_volterra, t_span, z0, dense_output=True)
t = np.linspace(0, 50, 501)
x_vals, y_vals = sol.sol(t)


In [ ]:
noise_level = 0.1
x_noisy = x_vals + noise_level*np.random.randn(len(t))
y_noisy = y_vals + noise_level*np.random.randn(len(t))

In [15]:
# dataframe
import pandas as pd

data = {'x': x_noisy,'y': y_noisy}
df = pd.DataFrame(data)
df

,x,y
0,10.072324,4.983736
1,10.355431,5.163191
2,11.130021,5.460713
3,11.873407,5.830131
4,12.306109,6.288585
...,...,...
496,8.597882,7.172516
497,8.193533,6.802143
498,7.741399,6.700107
499,7.076300,6.658352


In [10]:
import networkx as nx

variables = ['x', 'y']
time_steps = range(1, 6)

G_multivariate = nx.DiGraph()

# Add nodes for each variable and time step
for var in variables:
    for t in time_steps:
        G_multivariate.add_node((var, t))

# Add edges for causal influences:
# For each t>1, x_t depends on x_(t-1) and y_(t-1)
# Similarly, y_t depends on x_(t-1) and y_(t-1)

for t in time_steps:
    if t > 1:
        G_multivariate.add_edge(('x', t-1), ('x', t))
        G_multivariate.add_edge(('y', t-1), ('x', t))
        G_multivariate.add_edge(('x', t-1), ('y', t))
        G_multivariate.add_edge(('y', t-1), ('y', t))

print("Multivariate edges:")
for edge in G_multivariate.edges():
    print(edge)


Multivariate edges:
(('x', 1), ('x', 2))
(('x', 1), ('y', 2))
(('x', 2), ('x', 3))
(('x', 2), ('y', 3))
(('x', 3), ('x', 4))
(('x', 3), ('y', 4))
(('x', 4), ('x', 5))
(('x', 4), ('y', 5))
(('y', 1), ('x', 2))
(('y', 1), ('y', 2))
(('y', 2), ('x', 3))
(('y', 2), ('y', 3))
(('y', 3), ('x', 4))
(('y', 3), ('y', 4))
(('y', 4), ('x', 5))
(('y', 4), ('y', 5))


## Multivariate Case 

While the classic LV model is two-dimensional, we can easily create a multivariate time series by introducing multiple prey and/or predator species. For example, a three-dimensional system might look like:

$$
\frac{dx}{dt} = \alpha_1 x - \beta_{11} x y_1 - \beta_{12} x y_2
$$
$$
\frac{dy_1}{dt} = \delta_{1} x y_1 - \gamma_1 y_1
$$
$$
\frac{dy_2}{dt} = \delta_{2} x y_2 - \gamma_2 y_2
$$

Here:
- $ x $ is a common prey species.
- $ y_1 $ and $ y_2 $ are two different predator species that depend on $ x $ as a food source.
- The causal structure is now more complex: prey influences both predators, and both predators influence the prey. Depending on parameters, you might have partial causal independence between $ y_1 $ and $ y_2 $.

Or, you can have multiple prey species and one predator. For example:

$$
\frac{dx_1}{dt} = \alpha_1 x_1 - \beta_{1} x_1 y
$$
$$
\frac{dx_2}{dt} = \alpha_2 x_2 - \beta_{2} x_2 y
$$
$$
\frac{dy}{dt} = \delta_{1} x_1 y + \delta_{2} x_2 y - \gamma y
$$

Now, predator $ y $ is influenced by both $ x_1 $ and $ x_2 $, while each prey species is influenced by the common predator. This yields a three-variable time series with a known underlying causal graph.

In [12]:
import numpy as np
from scipy.integrate import solve_ivp

# Define parameters
alpha1, alpha2, beta1, beta2, delta1, delta2, gamma = 1.0, 0.5, 0.1, 0.1, 0.075, 0.075, 1.0

def lotka_volterra(t, z):
    x1, x2, y = z
    dx1dt = alpha1*x1 - beta1*x1*y
    dx2dt = alpha2*x2 - beta2*x2*y
    dydt = delta1*x1*y + delta2*x2*y - gamma*y
    return [dx1dt, dx2dt, dydt]

# Time span and initial conditions
t_span = (0, 50)
z0 = [10, 10, 5]  # initial conditions for x and y

# Solve ODE
sol = solve_ivp(lotka_volterra, t_span, z0, dense_output=True)
t = np.linspace(0, 50, 501)
x1_vals, x2_vals, y_vals = sol.sol(t)


In [13]:
noise_level = 0.1
x1_noisy = x1_vals + noise_level*np.random.randn(len(t))
x2_noisy = x2_vals + noise_level*np.random.randn(len(t))
y_noisy = y_vals + noise_level*np.random.randn(len(t))

In [14]:
# dataframe
import pandas as pd

data = {'x1': x1_noisy, 'x2': x2_noisy, 'y': y_noisy}
df = pd.DataFrame(data)
df

,x1,x2,y
0,9.955255,10.107982,4.983736
1,10.566165,9.991501,5.163191
2,11.009795,9.809308,5.460713
3,11.502416,9.887026,5.830131
4,11.874045,9.757248,6.288585
...,...,...,...
496,8.908662,-0.062406,7.172516
497,9.312600,0.063048,6.802143
498,9.572149,0.048897,6.700107
499,10.000065,0.270550,6.658352


In [16]:
import networkx as nx

variables = ['x1', 'x2', 'y']
time_steps = range(1, 6)

G_multivariate = nx.DiGraph()

# Add nodes for each variable and time step
for var in variables:
    for t in time_steps:
        G_multivariate.add_node((var, t))

# Add edges for causal influences:
# For each t>1, x_t depends on x_(t-1) and y_(t-1)
# Similarly, y_t depends on x_(t-1) and y_(t-1)

for t in time_steps:
    if t > 1:
        G_multivariate.add_edge(('x1', t-1), ('x1', t))
        G_multivariate.add_edge(('x2', t-1), ('x2', t))
        G_multivariate.add_edge(('y', t-1), ('x1', t))
        G_multivariate.add_edge(('y', t-1), ('x2', t))
        G_multivariate.add_edge(('x1', t-1), ('y', t))
        G_multivariate.add_edge(('x2', t-1), ('y', t))
        G_multivariate.add_edge(('y', t-1), ('y', t))

print("Multivariate edges:")
for edge in G_multivariate.edges():
    print(edge)


Multivariate edges:
(('x1', 1), ('x1', 2))
(('x1', 1), ('y', 2))
(('x1', 2), ('x1', 3))
(('x1', 2), ('y', 3))
(('x1', 3), ('x1', 4))
(('x1', 3), ('y', 4))
(('x1', 4), ('x1', 5))
(('x1', 4), ('y', 5))
(('x2', 1), ('x2', 2))
(('x2', 1), ('y', 2))
(('x2', 2), ('x2', 3))
(('x2', 2), ('y', 3))
(('x2', 3), ('x2', 4))
(('x2', 3), ('y', 4))
(('x2', 4), ('x2', 5))
(('x2', 4), ('y', 5))
(('y', 1), ('x1', 2))
(('y', 1), ('x2', 2))
(('y', 1), ('y', 2))
(('y', 2), ('x1', 3))
(('y', 2), ('x2', 3))
(('y', 2), ('y', 3))
(('y', 3), ('x1', 4))
(('y', 3), ('x2', 4))
(('y', 3), ('y', 4))
(('y', 4), ('x1', 5))
(('y', 4), ('x2', 5))
(('y', 4), ('y', 5))
